# 🚀 Render Remotion na Nuvem (Google Colab) + Envio Automático para o Telegram

Este notebook renderiza o vídeo **"Einstein e o GPS"** (1080x1920 @ 30fps, 3.400 frames) usando o poder de processamento da nuvem do Google e envia o arquivo final `.mp4` diretamente para o seu bot do Telegram.

### 1. Configuração do Bot do Telegram
Preencha as suas credenciais do Telegram abaixo:

In [ ]:
#@title 🔑 Credenciais do Telegram
TELEGRAM_BOT_TOKEN = "" #@param {type:"string"}
TELEGRAM_CHAT_ID = "" #@param {type:"string"}

assert TELEGRAM_BOT_TOKEN, "Por favor, insira o TELEGRAM_BOT_TOKEN"
assert TELEGRAM_CHAT_ID, "Por favor, insira o TELEGRAM_CHAT_ID"
print("✓ Credenciais configuradas com sucesso!")

### 2. Instalação do Ambiente (Node.js 20, Chrome & Dependências)

In [ ]:
# Instalar Node.js 20 LTS e dependências do Chromium
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs ffmpeg libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2
!node -v && npm -v

### 3. Upload ou Extração do Projeto Remotion

In [ ]:
# Se o projeto for enviado como zip (project.zip) via Google Drive ou painel lateral:
import os, zipfile

if os.path.exists("project.zip"):
    print("Extraindo project.zip...")
    with zipfile.ZipFile("project.zip", 'r') as zip_ref:
        zip_ref.extractall(".")
    print("✓ Extração concluída!")
else:
    print("Faça o upload do seu 'project.zip' no painel de arquivos à esquerda do Colab!")

!npm install

### 4. Renderização do Vídeo Completo (3.400 frames)

In [ ]:
!mkdir -p out
!npx remotion render Main out/einstein_gps_master.mp4 --crf=21 --gl=angle --concurrency=100%

import os
size_mb = os.path.getsize("out/einstein_gps_master.mp4") / (1024 * 1024)
print(f"✓ Vídeo renderizado com sucesso: {size_mb:.2f} MB")

### 5. Envio Automático para o Telegram

In [ ]:
import requests

video_path = "out/einstein_gps_master.mp4"
url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendVideo"

print(f"Enviando {video_path} para o Telegram...")
with open(video_path, 'rb') as video_file:
    response = requests.post(
        url,
        data={
            "chat_id": TELEGRAM_CHAT_ID,
            "caption": "🎬 Vídeo Renderizado: Einstein e o GPS\n📐 1080x1920 (9:16) @ 30 FPS\n⏱️ 3.400 frames (1m53s)\n✨ 100% Three.js Spline Clay",
            "supports_streaming": True
        },
        files={"video": video_file}
    )

if response.status_code == 200:
    print("🚀 VÍDEO ENTREGUE NO TELEGRAM COM SUCESSO!")
else:
    print(f"Erro no envio: {response.status_code} - {response.text}")